# Bag of Words

This notebook covers the Bag of Words (BoW) model, one of the simplest and most foundational text representation techniques in natural language processing.

The Bag of Words model represents text as a vector of word counts, ignoring grammar and word order but keeping track of word frequencies. This approach is the basis for many traditional NLP applications including text classification and information retrieval.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter

print("Libraries imported successfully.")

## 1. Understanding the Bag of Words Model

The Bag of Words model converts text into numerical vectors based on word counts. Each document is represented as a "bag" of words, where the order of words is disregarded but their frequencies are preserved.

Key concepts:
- **Vocabulary**: The unique set of all words across all documents
- **Document**: Any piece of text (sentence, paragraph, article)
- **Vector**: Numerical representation of a document based on word frequencies

In [ ]:
# Sample corpus for demonstration
corpus = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
]

print("=== Sample Corpus ===")
for i, doc in enumerate(corpus, 1):
    print(f"Document {i}: {doc}")

In [ ]:
# Manual implementation of Bag of Words
def bag_of_words_manual(documents):
    """
    Manually implement Bag of Words model.
    
    Parameters:
    -----------
    documents : list
        List of text documents
    
    Returns:
    --------
    tuple
        (vocabulary, document_vectors)
    """
    # Build vocabulary from all documents
    all_words = []
    for doc in documents:
        words = doc.lower().split()
        # Remove punctuation
        words = [word.rstrip('.,!?;') for word in words]
        all_words.extend(words)
    
    # Get unique words (vocabulary)
    vocabulary = sorted(set(all_words))
    
    # Create document vectors
    document_vectors = []
    for doc in documents:
        words = doc.lower().split()
        words = [word.rstrip('.,!?;') for word in words]
        word_counts = Counter(words)
        
        # Create vector of word counts
        vector = [word_counts.get(word, 0) for word in vocabulary]
        document_vectors.append(vector)
    
    return vocabulary, document_vectors

# Apply manual BoW
vocabulary, vectors = bag_of_words_manual(corpus)

print("=== Vocabulary ===")
print(f"Vocabulary size: {len(vocabulary)}")
print(f"Words: {vocabulary}\n")

print("=== Document Vectors ===")
for i, vec in enumerate(vectors, 1):
    print(f"Document {i}: {vec}")

## 2. Using Scikit-learn's CountVectorizer

Scikit-learn provides a convenient CountVectorizer class that implements the Bag of Words model with many useful options.

In [ ]:
# Using CountVectorizer from scikit-learn
vectorizer = CountVectorizer()

# Fit and transform the corpus
bow_matrix = vectorizer.fit_transform(corpus)

print("=== Using CountVectorizer ===")
print(f"Vocabulary: {vectorizer.get_feature_names_out()}")
print(f"\nBoW Matrix shape: {bow_matrix.shape}")
print(f"\nBoW Matrix (dense):\n{bow_matrix.toarray()}")

In [ ]:
# Convert to DataFrame for better visualisation
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("=== Bag of Words DataFrame ===")
print(bow_df)

## 3. Customising CountVectorizer

The CountVectorizer offers many options to customise the Bag of Words representation:
- **lowercase**: Convert all text to lowercase (default: True)
- **stop_words**: Remove common English stop words
- **max_features**: Limit vocabulary size
- **ngram_range**: Include n-grams (e.g., bigrams)
- **min_df**: Ignore terms that appear in fewer than this many documents

In [ ]:
# Customising CountVectorizer with various options

print("=== Customising CountVectorizer ===\n")

# Example 1: With stop words removal
vectorizer_no_stop = CountVectorizer(stop_words='english')
bow_no_stop = vectorizer_no_stop.fit_transform(corpus)
print("1. With stop words removed:")
print(f"   Vocabulary: {vectorizer_no_stop.get_feature_names_out()}\n")

# Example 2: With minimum document frequency
vectorizer_min_df = CountVectorizer(min_df=2)
bow_min_df = vectorizer_min_df.fit_transform(corpus)
print("2. With min_df=2 (words must appear in at least 2 documents):")
print(f"   Vocabulary: {vectorizer_min_df.get_feature_names_out()}\n")

# Example 3: With bigrams
vectorizer_bigram = CountVectorizer(ngram_range=(1, 2))
bow_bigram = vectorizer_bigram.fit_transform(corpus)
print("3. With bigrams (ngram_range=(1, 2)):")
print(f"   Vocabulary: {vectorizer_bigram.get_feature_names_out()}")

## 4. Practical Application: Text Classification

Let's see how Bag of Words can be used for a simple text classification task.

In [ ]:
# Sample dataset for classification
texts = [
    "I love this product, it is amazing!",
    "Great quality and fast delivery.",
    "Absolutely fantastic, highly recommend!",
    "Terrible product, complete waste of money.",
    "Very disappointed with this purchase.",
    "Worst experience ever, do not buy.",
]
labels = [1, 1, 1, 0, 0, 0]  # 1 = positive, 0 = negative

# Create BoW representation
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)
y = np.array(labels)

print("=== Text Classification with BoW ===")
print(f"\nText data: {len(texts)} documents")
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")
print(f"Feature matrix shape: {X.shape}")

print("\n--- Sample vectors ---")
for i in range(min(3, len(texts))):
    print(f"\nText: '{texts[i]}'")
    print(f"Label: {'Positive' if labels[i] == 1 else 'Negative'}")
    print(f"Vector (non-zero elements): {dict(zip(vectorizer.get_feature_names_out(), X[i].toarray()[0]))}")

## 5. Limitations of Bag of Words

While the Bag of Words model is simple and effective, it has several limitations:

1. **Loss of Word Order**: The model ignores grammar and word order, which can be important for meaning.

2. **Vocabulary Size**: Large documents can create very high-dimensional vectors.

3. **No Semantic Meaning**: Words with similar meanings are treated as completely different tokens.

4. **Sparse Representations**: Most elements in the vector are zero, making storage inefficient.

These limitations led to the development of more sophisticated techniques like TF-IDF and word embeddings.

In [ ]:
# Demonstrate sparsity
total_elements = bow_matrix.shape[0] * bow_matrix.shape[1]
non_zero = bow_matrix.nnz  # number of non-zero elements
sparsity = (1 - non_zero / total_elements) * 100

print("=== Sparsity Analysis ===")
print(f"Total elements in matrix: {total_elements}")
print(f"Non-zero elements: {non_zero}")
print(f"Sparsity: {sparsity:.2f}%")
print(f"\nThis high sparsity is typical of Bag of Words representations,")
print("which is why more efficient representations like TF-IDF are often preferred.")

## Summary

In this notebook, we covered the Bag of Words model:

1. **Manual Implementation**: Built BoW from scratch to understand the underlying concept
2. **CountVectorizer**: Used scikit-learn's implementation for convenience
3. **Customisation**: Explored various options like stop words removal, min_df, and n-grams
4. **Classification Application**: Demonstrated how BoW can be used for text classification
5. **Limitations**: Discussed the shortcomings of the BoW model

The Bag of Words model is a foundational technique that paved the way for more advanced text representation methods. While simple, it remains useful for many NLP tasks and serves as a baseline for comparing more sophisticated approaches.